In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple

In [2]:
def load_forecast_csv(path: str) -> pd.DataFrame:

    # Try standard read; if headers exist we'll map by position, else name them.
    df = pd.read_csv(path)
    # If the file has fewer than 3 columns, try reading with no header and explicit names
    if df.shape[1] < 3:
        df = pd.read_csv(path, header=None)

    # After loading, select by position: B (index 1), C (index 2), D (index 3)
    # If not enough columns, raise an informative error
    if df.shape[1] < 4:
        # Some CSVs may include only 3 total columns (A,B,C), where Date is B (1), True is C (2), Pred is D (3) missing.
        # But the user's format specifies 4 columns total with A unused. If that's not the case, fallback:
        # Try using the last three columns as (Date, True, Pred).
        if df.shape[1] >= 3:
            use = df.iloc[:, -3:]
        else:
            raise ValueError(f"{path} must have at least 3 columns; got {df.shape[1]}")
    else:
        use = df.iloc[:, [1,2,3]]

    use = use.copy()
    use.columns = ['Date', 'True', 'Pred']
    use['Date'] = pd.to_datetime(use['Date'], errors='coerce')
    use = use.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
    # Coerce numeric
    use['True'] = pd.to_numeric(use['True'], errors='coerce')
    use['Pred'] = pd.to_numeric(use['Pred'], errors='coerce')
    use = use.dropna(subset=['True', 'Pred']).reset_index(drop=True)
    return use

In [3]:
def cumulative_sse_difference(df_a: pd.DataFrame, df_b: pd.DataFrame) -> pd.DataFrame:
    """
    Given two aligned DataFrames with columns ['Date','True','Pred'],
    compute the per-date squared error for each, then the difference (SE_A - SE_B),
    and the cumulative sum over time. Returns a DataFrame with:
    ['Date','SE_A','SE_B','Diff','CumDiff']
    """
    # Align by inner join on Date
    A = df_a[['Date','True','Pred']].rename(columns={'Pred':'Pred_A'})
    B = df_b[['Date','True','Pred']].rename(columns={'Pred':'Pred_B'})
    merged = pd.merge(A, B[['Date','Pred_B']], on='Date', how='inner')

    # If True values differ between files (they shouldn't), keep A's True and warn via comment
    merged['SE_A'] = (merged['True'] - merged['Pred_A'])**2
    merged['SE_B'] = (merged['True'] - merged['Pred_B'])**2
    merged['Diff'] = merged['SE_A'] - merged['SE_B']
    merged['CumDiff'] = merged['Diff'].cumsum()
    return merged[['Date','SE_A','SE_B','Diff','CumDiff']]

In [4]:
def plot_cumulative_difference(result_df: pd.DataFrame, title: str = "Cumulative SSE Difference") -> None:
    plt.figure()
    plt.plot(result_df['Date'], result_df['CumDiff'])
    plt.axhline(0, linestyle='--')
    plt.xlabel('Date')
    plt.ylabel('Cumulative SSE Difference (Model A - Model B)')
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [13]:
# Set file paths here
file_a = 'HAR.csv'         # baseline model
file_b = 'HAR BIC.csv'     # comparison model

# Labels for the legend/title
label_a = 'HAR'
label_b = 'HAR BIC'


In [ ]:
df_a = load_forecast_csv(file_a)
df_b = load_forecast_csv(file_b)

# Compute cumulative difference
res = cumulative_sse_difference(df_a, df_b)

# Quick summary
final_sse_a = res['SE_A'].sum()
final_sse_b = res['SE_B'].sum()
final_cumdiff = res['CumDiff'].iloc[-1]

print(f"Total SSE ({label_a}): {final_sse_a:,.6f}")
print(f"Total SSE ({label_b}): {final_sse_b:,.6f}")
print(f"Final Cumulative Difference (A - B): {final_cumdiff:,.6f}")

# Plot
plot_cumulative_difference(res, title=f"Cumulative SSE Difference: {label_a} - {label_b}")